## Hyper-Parameter Optimization


---

### 1. Key Hyperparameters to Tune

For a feedforward neural network (MLP) on Fashion MNIST:

* **Number of hidden layers**
* **Number of neurons per layer**
* **Activation function**
* **Optimizer** (Adam, RMSProp, SGD, etc.)
* **Learning rate**
* **Batch size**
* **Dropout rate** (optional, helps with regularization)
* **Number of epochs**

---

### 2. Suggested Parameter Grid for Bayesian Optimization

Since Bayesian search works better with **continuous ranges** than with very large discrete sets, define ranges carefully:

```python
from skopt.space import Real, Integer, Categorical

param_grid = {
    # Architecture
    'num_layers': Integer(1, 5),                        # 1–5 hidden layers
    'num_neurons': Integer(32, 512),                    # neurons per layer
    'activation': Categorical(['relu', 'tanh', 'sigmoid']),
    
    # Optimization
    'optimizer': Categorical(['adam', 'rmsprop', 'sgd']),
    'learning_rate': Real(1e-4, 1e-1, prior='log-uniform'),
    
    # Training
    'batch_size': Integer(32, 256),
    'epochs': Integer(5, 30),
    
    # Regularization
    'dropout_rate': Real(0.0, 0.5)
}
```

---

### 3. Practical Notes

* **Log-uniform for learning rate** → ensures better exploration across magnitudes.
* **Neurons**: Instead of searching independently for each layer, you can start with a fixed width (`num_neurons`) for all hidden layers. Later, if needed, expand to per-layer neuron counts.
* **Epochs**: Use **early stopping** during training. That way, the search doesn’t waste time if the model stops improving.
* **Batch size**: Don’t go too high; Fashion MNIST works well with 64–128.
* **Dropout**: Optional but improves generalization.
* **Activation**: Most Fashion MNIST models work best with `relu`, but it’s good to test `tanh` too.

---

### 4. Example Usage with BayesSearchCV

```python
from skopt import BayesSearchCV
from sklearn.model_selection import StratifiedKFold

# Suppose you wrap your Keras/TensorFlow model in a scikit-learn compatible estimator
opt = BayesSearchCV(
    estimator=my_tf_classifier,    # your custom model wrapper
    search_spaces=param_grid,
    n_iter=30,                     # number of search iterations
    cv=StratifiedKFold(3),         # 3-fold CV
    n_jobs=-1,
    verbose=1
)

opt.fit(X_train, y_train)
print(opt.best_params_)
print(opt.best_score_)
```

---

## For Fashion MNIST Dataset


---

### 1. Likely Parameter Grid He Used

Since the final model is:

* **Layers**: 2
* **Neurons**: 300 and 100
* **Activation**: relu
* **Optimizer**: SGD
* **Epochs**: 30
* **Output**: 10 (fixed, since Fashion MNIST has 10 classes)

A reasonable `param_grid` that would allow the optimizer to discover this is:

```python
from skopt.space import Real, Integer, Categorical

param_grid = {
    'num_layers': Integer(1, 3),                        # allows up to 3 hidden layers
    'num_neurons_layer1': Integer(64, 512),             # search around 64–512
    'num_neurons_layer2': Integer(32, 256),             # smaller second layer
    'activation': Categorical(['relu', 'tanh']),
    'optimizer': Categorical(['sgd', 'adam', 'rmsprop']),
    'learning_rate': Real(1e-4, 1e-1, prior='log-uniform'),
    'batch_size': Integer(32, 256),
    'epochs': Integer(10, 50)                           # 30 is inside the range
}
```

👉 This search space **covers his final architecture** but still allows exploration.
The optimizer must have selected:

* `num_layers=2`
* `num_neurons_layer1=300`
* `num_neurons_layer2=100`
* `activation=relu`
* `optimizer=sgd`
* `epochs=30`

---

### 2. BayesSearchCV Code

He probably wrapped the Keras model in a scikit-learn compatible class.
Here’s a minimal structure:

```python
from sklearn.base import BaseEstimator, ClassifierMixin
import tensorflow as tf
from tensorflow import keras

# Custom wrapper
class MyKerasClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, num_layers=2, num_neurons_layer1=128, num_neurons_layer2=64, activation='relu', optimizer='adam', learning_rate=0.001, batch_size=128, epochs=20):
        self.num_layers = num_layers
        self.num_neurons_layer1 = num_neurons_layer1
        self.num_neurons_layer2 = num_neurons_layer2
        self.activation = activation
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs

    def build_model(self):
        model = keras.models.Sequential()
        model.add(keras.layers.Flatten(input_shape=[28, 28]))
        
        # Add first hidden layer
        model.add(keras.layers.Dense(self.num_neurons_layer1, activation=self.activation))
        
        # Add second hidden layer if chosen
        if self.num_layers >= 2:
            model.add(keras.layers.Dense(self.num_neurons_layer2, activation=self.activation))
        
        # Add third hidden layer if chosen
        if self.num_layers >= 3:
            model.add(keras.layers.Dense(self.num_neurons_layer2 // 2, activation=self.activation))
        
        # Output layer (10 classes for Fashion MNIST)
        model.add(keras.layers.Dense(10, activation="softmax"))

        opt = None
        if self.optimizer == 'sgd':
            opt = keras.optimizers.SGD(learning_rate=self.learning_rate)
        elif self.optimizer == 'adam':
            opt = keras.optimizers.Adam(learning_rate=self.learning_rate)
        elif self.optimizer == 'rmsprop':
            opt = keras.optimizers.RMSprop(learning_rate=self.learning_rate)

        model.compile(loss="sparse_categorical_crossentropy",
                      optimizer=opt,
                      metrics=["accuracy"])
        return model

    def fit(self, X, y):
        self.model_ = self.build_model()
        self.model_.fit(X, y, epochs=self.epochs,
                        batch_size=self.batch_size,
                        verbose=0)
        return self

    def predict(self, X):
        y_pred = self.model_.predict(X)
        return y_pred.argmax(axis=1)

    def score(self, X, y):
        return self.model_.evaluate(X, y, verbose=0)[1]  # return accuracy
```

---

### Bayesian Search

```python
from skopt import BayesSearchCV
from sklearn.model_selection import StratifiedKFold

search = BayesSearchCV(
    estimator=MyKerasClassifier(),
    search_spaces=param_grid,
    n_iter=20,                         # number of optimization trials
    cv=StratifiedKFold(3),             # 3-fold CV
    n_jobs=1,                          # TensorFlow doesn’t like multi-threading
    verbose=2
)

search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best accuracy:", search.best_score_)
```

---

✅ This setup would very likely give the same final hyperparameters as your teacher’s (`2 layers, 300–100 neurons, relu, sgd, 30 epochs`).
That means his `param_grid` must have been broad enough to include these values, and Bayesian optimization.

---